# 10c | Drills: Machine Learning

Covers **10a and 10b**: splitting, baselines, Pipelines, cross-validation, tuning, regularization, leakage, and the conceptual follow-ups interviewers like to ask. Roughly half code questions and half oral questions.

**How to use this lesson:** There are no input boxes; each problem is followed directly by its answer. For code questions, think about how you would write it first, then read the answer and its output. Questions marked 🗣 are oral: answer **out loud** in your own words first, then compare with the reference answer. After each one, cover the answer and try to restate the key points.

Code questions on this page run in order, so variables defined earlier can be reused later.

Every code question starts with this already run:

```python
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from practice_data import load_capstone

# Simulated binary classification data: 1500 rows, 8 features, about 20% positives
X, y = make_classification(n_samples=1500, n_features=8, n_informative=4,
                           weights=[0.8, 0.2], random_state=0)
```

### Problem 1 · Stratified split

Hold out 25% of `X`, `y` as a test set, keeping the positive rate the same on both sides.

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))
```

**Result:**

```text
(1125, 8) (375, 8)
0.205 0.205
```

> `stratify=y` gives the training and test sets the same positive rate. Always add it when classes are imbalanced.

### Problem 2 · The dumbest baseline

If the model always predicts 0 (the majority class), what is its accuracy on the test set?

```python
round(1 - y_test.mean(), 3)
```

**Result:**

```text
0.795
```

> Only 20% are positives, so always guessing 0 already gives 80% accuracy. A model with 85% accuracy is therefore not necessarily good; always compare any metric with a baseline first.

### Problem 3 · Pipeline + logistic regression

Chain standardization and logistic regression into a Pipeline, fit it on the training set, and compute the test ROC-AUC.

```python
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]
round(roc_auc_score(y_test, proba), 4)
```

**Result:**

```text
0.7577
```

> Column 1 of `predict_proba` is the probability of the positive class. With standardization inside the Pipeline, the mean and standard deviation are computed from the training set only.

### Problem 4 · Comparing three models with cross-validation

Use 5-fold cross-validation on the training set to compare the AUC of logistic regression, a random forest, and gradient boosting.

```python
candidates = {
    "logistic": LogisticRegression(max_iter=1000),
    "random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
for name, model in candidates.items():
    p = Pipeline([("scale", StandardScaler()), ("model", model)])
    s = cross_val_score(p, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:18s} {s.mean():.4f} ± {s.std():.4f}")
```

**Result:**

```text
logistic           0.7256 ± 0.0128
random forest      0.9418 ± 0.0220
gradient boosting  0.9309 ± 0.0233
```

> Tree models don't actually need standardization; it is in the same Pipeline only to keep the code uniform, and it doesn't affect the results. The signal in this data is nonlinear, so the tree models are clearly better; the user data in 10b was the opposite, where logistic regression was no worse than the trees. So always compare instead of assuming. Look at both the mean and the standard deviation: when the gap is smaller than one standard deviation, it's hard to say which is better.

### Problem 5 · Great on training, poor on validation 🗣

Training AUC is 0.99 and cross-validated AUC is 0.71. What does this tell you? What do you do next?

**Reference answer:**

- **What it means:** overfitting. The model also learned the noise in the training data; its variance is too high.
- **Rule out first:** you would suspect leakage if the validation score were much *higher* than expected; here training is high and validation low, which looks more like an overly complex model.
- **Next steps:** reduce complexity (shallower trees, larger minimum samples per leaf, stronger regularization); more data; bagging / ensembles; feature selection.
- **How to judge improvement:** look only at the cross-validated score, and at whether the gap between training and validation shrinks.

**Short version:** A large gap between training and validation AUC means overfitting, so I'd reduce model complexity, add regularization or more data, and judge each change by cross-validated score.

### Problem 6 · Training scores cannot be used for evaluation

Set the random forest's `min_samples_leaf` to 1 and to 30, and look at the training and test AUC for each.

```python
for leaf in [1, 30]:
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, random_state=0)
    rf.fit(X_train, y_train)
    auc_tr = roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1])
    auc_te = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    print(f"min_samples_leaf={leaf:2d}  train {auc_tr:.4f}  test {auc_te:.4f}")
```

**Result:**

```text
min_samples_leaf= 1  train 1.0000  test 0.9699
min_samples_leaf=30  train 0.9557  test 0.9543
```

> With leaves of size 1 the training AUC is 1.0, which only shows that the trees memorized the training data; training scores can't evaluate a model. Here the test AUC with leaf size 1 is actually slightly higher: the random forest keeps variance under control by averaging many trees, which a single deep tree can't do. Overfitting is judged by the validation / test score and its gap from the training score. We looked at the test set here only for demonstration; in a real workflow, tune this parameter with cross-validation.

### Problem 7 · Tuning regularization strength with GridSearchCV

Search logistic regression's `C` over `[0.01, 0.1, 1, 10]`, with 5 folds, selecting by AUC.

```python
gs = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    param_grid={"model__C": [0.01, 0.1, 1, 10]},
    cv=5,
    scoring="roc_auc",
)
gs.fit(X_train, y_train)
print(gs.best_params_, round(gs.best_score_, 4))
```

**Result:**

```text
{'model__C': 10} 0.7257
```

> Parameter names inside a Pipeline are `stepname__paramname` (two underscores). Smaller `C` means stronger regularization. `gs.best_estimator_` is the model refit on the full training set with the best parameters. Here the best `C` lands on the edge of the grid (10), which means the grid should be extended toward larger values; however, its score is almost the same as the untuned logistic regression in Problem 4, so regularization strength matters little on this data.

### Problem 8 · L1 sets coefficients to 0

Using the same Pipeline with L1 and then L2 regularization (`C=0.02`), count how many coefficients are exactly 0 in each case.

```python
for pen in ["l1", "l2"]:
    p = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(penalty=pen, solver="liblinear", C=0.02)),
    ])
    p.fit(X_train, y_train)
    coef = p.named_steps["model"].coef_[0]
    print(pen, coef.round(2), "number of zeros:", (coef == 0).sum())
```

**Result:**

```text
l1 [0.   0.21 0.24 0.   0.   0.   0.   0.  ] number of zeros: 6
l2 [-0.02  0.32  0.34  0.06  0.14 -0.08  0.1   0.01] number of zeros: 0
```

> L1 pushes unimportant coefficients to exactly 0, which amounts to automatic feature selection; L2 only shrinks coefficients and never makes them exactly 0.

### Problem 9 · L1 vs. L2 🗣

What is the difference between L1 and L2 regularization? When is each appropriate?

**Reference answer:**

- **Form:** L1 penalizes the sum of absolute values of the coefficients; L2 penalizes the sum of squares.
- **Effect:** L1 pushes some coefficients to exactly 0 (sparsity, built-in feature selection); L2 shrinks all coefficients toward 0 together but not to 0.
- **Correlated features:** with a group of highly correlated features, L1 tends to keep one and drop the others somewhat arbitrarily; L2 spreads the weight across them, which is more stable.
- **How to choose:** L1 when you want a small set of interpretable features; L2 when there are many correlated features and the main goal is stable prediction; Elastic Net when you want both. Choose the strength by cross-validation in every case.

**Short version:** L1 penalizes absolute values and sets some coefficients exactly to zero, giving feature selection; L2 penalizes squares and shrinks all coefficients smoothly, which is more stable with correlated features.

### Problem 10 · What target leakage looks like

Add a column "`y` plus a little noise" to `X`, simulating a feature determined by the outcome itself (e.g., using "was refunded" to predict "will place an order"), then run 5-fold cross-validation.

```python
rng = np.random.default_rng(0)
leak = y + rng.normal(0, 0.3, size=len(y))
X_leak = np.column_stack([X, leak])
pipe_leak = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
s = cross_val_score(pipe_leak, X_leak, y, cv=5, scoring="roc_auc")
round(s.mean(), 4)
```

**Result:**

```text
0.9909
```

> The AUC is close to 1. When results look too good to be true, your first reaction should be to check for leakage: for each feature, ask "At the moment of prediction, does this value already exist?"

### Problem 11 · Multiple rows per user 🗣

The data are daily records per user, with dozens of rows per user. What is wrong with a random train/test split? How should you split?

**Reference answer:**

- **The problem:** rows from the same user end up in both training and test sets, so the model can "recognize" the person and the test score is inflated; in production it faces new users and will do much worse.
- **How to split:** group by user so all of a user's rows are on one side (GroupKFold for cross-validation).
- **If there is also a time dimension:** split by time as well — train on earlier data, test on later data.
- **One-line principle:** the split should mimic the data the model will actually see after deployment.

**Short version:** Random splits put the same user in both train and test, which inflates the score; I'd split by user, with GroupKFold for cross-validation, and also by time if the use case is forecasting.

### Problem 12 · Building features from an event log

Using `events` from `load_capstone()`, count each user's `view`, `click`, and `search` events (`purchase` is off limits), left-join the counts onto `users`, and fill users with no records with 0.

```python
users, events = load_capstone()
feats = (
    events[events["event_type"] != "purchase"]
    .pivot_table(index="user_id", columns="event_type", values="event_date",
                 aggfunc="count", fill_value=0)
    .reset_index()
)
data = users.merge(feats, on="user_id", how="left")
data[["view", "click", "search"]] = data[["view", "click", "search"]].fillna(0)
print(data.shape)
data[["user_id", "country", "age", "acq_channel", "view", "click", "search", "converted"]].head()
```

**Result:**

```text
(8000, 11)
```

|   | user_id | country | age | acq_channel | view | click | search | converted |
|---|---|---|---|---|---|---|---|---|
| 0 | 1 | US | 45.0 | organic | 1.0 | 1.0 | 2.0 | 0 |
| 1 | 2 | US | 33.0 | organic | 3.0 | 1.0 | 3.0 | 0 |
| 2 | 3 | US | 44.0 | search | 4.0 | 1.0 | 0.0 | 0 |
| 3 | 4 | US | 55.0 | organic | 1.0 | 4.0 | 3.0 | 0 |
| 4 | 5 | US | 35.0 | search | 0.0 | 1.0 | 1.0 | 0 |

*5 rows × 8 columns*

> `purchase` is the conversion itself, so including it is leakage. `revenue_30d` can't be a feature either: it only exists after conversion. Strictly speaking, `view`, `click`, and `search` have a similar problem: in this practice dataset they cover the 30 days after signup, the same window as conversion, so some actions may have happened after conversion. In a real project, first fix a "prediction time" (e.g., day 3 after signup) and build features only from actions before it. The three columns are decimals (`1.0`) because the `merge` produced missing values; after filling with 0 the dtype is still float.

### Problem 13 · ColumnTransformer: handling numeric and categorical columns separately

Median-impute and standardize the numeric columns, one-hot encode the categorical columns, follow with logistic regression, and run 5-fold cross-validation on an 80% training set.

```python
num_cols = ["age", "treatment", "view", "click", "search"]
cat_cols = ["country", "acq_channel"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    data[num_cols + cat_cols], data["converted"],
    test_size=0.2, stratify=data["converted"], random_state=0,
)
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_c = Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=1000))])
s = cross_val_score(pipe_c, Xc_train, yc_train, cv=5, scoring="roc_auc")
print(s.round(4), round(s.mean(), 4))
```

**Result:**

```text
[0.7631 0.7774 0.7635 0.7649 0.7524] 0.7643
```

> `handle_unknown="ignore"`: a category not seen in training won't raise an error at prediction time. With all preprocessing inside the Pipeline, each cross-validation fold computes the medians and category lists from its training part only.

### Problem 14 · Reading coefficients: which features matter most

Fit `pipe_c` on the training set, extract the feature names after preprocessing and the logistic regression coefficients, and sort by absolute value, largest first.

```python
pipe_c.fit(Xc_train, yc_train)
names = pipe_c.named_steps["pre"].get_feature_names_out()
coefs = pipe_c.named_steps["model"].coef_[0]
pd.Series(coefs, index=names).sort_values(key=abs, ascending=False).round(3)
```

**Result:**

|   |   |
|---|---|
| num__view | 0.689 |
| num__click | 0.506 |
| cat__country_UK | -0.486 |
| cat__acq_channel_social | -0.486 |
| cat__acq_channel_search | -0.457 |
| cat__acq_channel_referral | -0.434 |
| num__search | 0.423 |
| cat__country_DE | -0.411 |
| cat__country_US | -0.38 |
| cat__country_CA | -0.217 |
| num__age | 0.197 |
| cat__acq_channel_organic | -0.118 |
| num__treatment | -0.084 |

*13 rows × 1 column*

> The numeric features are standardized, so their coefficients are directly comparable: the change in log-odds per one-standard-deviation increase. The 0/1 columns of categorical features are on a different scale, so don't compare them directly with numeric features. Also, `OneHotEncoder` keeps one column per category by default (no reference category), so the sign of a single category's coefficient is meaningless (here all countries and channels are negative); only differences within the same variable are meaningful, e.g., `organic` is about 0.37 higher than `social`.

### Problem 15 · Monotone transformations of a feature 🗣

If you replace `age` with `log(age)`, do a random forest's predictions change? What about logistic regression?

**Reference answer:**

- **Random forest / gradient boosting:** essentially unchanged. Trees only use the ordering of feature values to find split points, and a monotone transformation preserves the ordering.
- **Logistic / linear regression:** they change. The model assumes the log-odds are linear in the feature, and `age` and `log(age)` correspond to different functional forms.
- **By the same logic:** tree models don't need standardization; kNN, SVM, regularized linear models, and PCA do.

**Short version:** Tree models only use the ordering of a feature, so a monotone transform barely changes them, while linear and logistic models change because the assumed functional form changes.

### Problem 16 · High-cardinality categorical features 🗣

A "city" feature has 2,000 distinct values. How do you encode it? What should you watch out for?

**Reference answer:**

- **The problem with one-hot:** 2,000 very sparse columns; many cities have only a few rows, so their coefficients are poorly estimated.
- **Common approaches:** ① frequency encoding (how often the city appears); ② target encoding (the city's average conversion rate); ③ merging rare cities into "other"; ④ a model that handles categories natively (LightGBM, CatBoost).
- **The target-encoding trap:** it uses y, so it must be computed within the training part of each cross-validation fold and shrunk toward the overall mean for cities with few samples; otherwise it leaks and overfits badly.
- **New cities at test time:** need a default (the overall mean or the "other" category).

**Short version:** One-hot would be huge and sparse, so I'd use frequency or target encoding or group rare levels. Target encoding must be computed within each training fold and shrunk toward the global mean to avoid leakage.

### Problem 17 · Class imbalance 🗣

Positives are only 2%. What do you need to watch out for in training and in evaluation?

**Reference answer:**

- **Evaluation:** don't look at accuracy (guessing all negatives gives 98%); use PR-AUC, precision / recall, or a business metric like precision@k; ROC-AUC is optimistic under extreme imbalance.
- **Splitting:** stratify so every fold has enough positives.
- **Training:** first try no special treatment and just tune the threshold; then try `class_weight="balanced"` or reweighting positives; do any over- / undersampling on the training set only.
- **Threshold:** the default 0.5 is often wrong; choose it from the costs of the two error types.
- **Caveat:** reweighting or resampling miscalibrates the predicted probabilities; recalibrate if you need the probabilities themselves.

**Short version:** I'd avoid accuracy and use PR-AUC or precision and recall, stratify the splits, try class weights or resampling on the training data only, and choose the threshold from error costs, recalibrating probabilities if needed.

### Problem 18 · Why does a random forest pick features at random? 🗣

At each split, a random forest considers only a random subset of features. Why?

**Reference answer:**

- **Purpose:** reduce the correlation between trees.
- **Reason:** the variance of an average of B trees is roughly ρσ² + (1−ρ)σ²/B. More trees shrink the second term, but the first term is set by the correlation ρ and doesn't fall as trees are added.
- **Without random feature selection:** if there is one very strong feature, every tree splits on it first, the trees look alike, and averaging helps little.
- **The cost:** each individual tree gets slightly worse (bias rises a bit), but overall variance falls by more.

**Short version:** Feature subsampling decorrelates the trees. The variance of an average of correlated trees is bounded by the correlation term, so making the trees less similar is what lets averaging reduce variance.

### Problem 19 · Are more trees always better? 🗣

Does adding more trees to a random forest cause overfitting? What about gradient boosting?

**Reference answer:**

- **Random forest:** more trees don't cause overfitting. They only make the average more stable, and the error converges to a limit; the cost is computation. Overfitting is controlled by tree depth and leaf size.
- **Gradient boosting:** yes. Each new tree fits the remaining residuals, so too many trees start fitting noise. Use a small learning rate plus early stopping and choose the number of trees by validation performance.
- **In one line:** in bagging, adding trees reduces variance; in boosting, adding trees reduces bias, and reducing it too far is overfitting.

**Short version:** More trees don't overfit a random forest; they only stabilize the average. In boosting each tree fits residuals, so too many trees overfit, which is why we use a small learning rate with early stopping.

### Problem 20 · PCA without standardization 🗣

Two features: annual income (in dollars, from tens of thousands to hundreds of thousands) and age (in years). What happens if you run PCA without standardizing?

**Reference answer:**

- **Result:** the first principal component is almost exactly the "income" direction. PCA looks for the direction of maximum variance, and the variance of income (in dollars) is millions of times larger than that of age.
- **The problem:** the result only reflects the choice of units — switch to "thousands of dollars" and the conclusion changes — so it is meaningless.
- **What to do:** standardize first when units aren't comparable (equivalent to PCA on the correlation matrix); skip standardization only when units are genuinely comparable (e.g., all the same kind of measurement).
- **Related issue:** k-means and kNN are also distance-based and are likewise dominated by large-unit features.

**Short version:** PCA maximizes variance, so without scaling the first component is essentially income, purely because of its units. Standardize first unless the features share a meaningful common scale; the same applies to k-means and kNN.

## Syntax recap

| Goal | Code |
|---|---|
| Stratified split | `train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)` |
| Baseline | Always guess the majority class: `1 - y_test.mean()` |
| Standardization + model | `Pipeline([("scale", StandardScaler()), ("model", ...)])` |
| Separate numeric / categorical handling | `ColumnTransformer([("num", ..., num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])` |
| Cross-validation | `cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")` |
| Tuning | `GridSearchCV(pipe, {"model__C": [...]}, cv=5, scoring="roc_auc")` |
| Regularization | `LogisticRegression(penalty="l1", solver="liblinear", C=...)`; coefficients: `named_steps["model"].coef_[0]` |
| Probabilities / AUC | `predict_proba(X)[:, 1]`, `roc_auc_score(y, proba)` |
| Feature names | `named_steps["pre"].get_feature_names_out()` |

The common thread in the oral questions: **make the split mimic deployment, put preprocessing in a Pipeline, check for leakage when results look too good to be true, and match model complexity to the signal.**